# Module 1 — Data Pipeline
## Zepto Data & AI Platform Capstone

**Pipeline**: scrape → clean → convert (GBP → INR) → store (SQLite) → query (SQL + pandas)

**Data source**: [books.toscrape.com](https://books.toscrape.com) — a public scraping-practice site. No login, no API key, no paid tier.

**Fixed currency rate**: `1 GBP = 105.50 INR` (project-defined constant, no API, no date reference)

> **How to run**: `Kernel → Restart Kernel and Run All Cells`

---

## Step 1 — Imports and Configuration

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time

print('Libraries imported successfully.')
print(f'pandas version : {pd.__version__}')

Libraries imported successfully.
pandas version : 2.2.2


In [2]:
# Fixed currency rate — project-defined constant, no API, no date reference
GBP_TO_INR = 105.50

CATALOGUE_BASE = 'https://books.toscrape.com/catalogue/'

CATEGORY_URLS = [
    ('Mystery', 'https://books.toscrape.com/catalogue/category/books/mystery_3/'),
    ('Science', 'https://books.toscrape.com/catalogue/category/books/science_22/'),
    ('Romance', 'https://books.toscrape.com/catalogue/category/books/romance_8/'),
    ('History', 'https://books.toscrape.com/catalogue/category/books/history_32/'),
    ('Travel',  'https://books.toscrape.com/catalogue/category/books/travel_2/'),
]

MIN_BOOKS      = 60
MIN_CATEGORIES = 3
RATING_MAP     = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
DB_PATH        = 'books.db'

print(f'GBP_TO_INR = {GBP_TO_INR}  |  DB = {DB_PATH}')

GBP_TO_INR = 105.5  |  DB = books.db


---
## Step 2 — Scrape books.toscrape.com

**Goal**: ≥ 60 books across ≥ 3 categories. No login, no API key.

| Field | HTML source | Raw example |
|-------|------------|-------------|
| `title` | `article.h3 > a[title]` | `"Sharp Objects"` |
| `price_raw` | `p.price_color` text | `"Â£47.82"` |
| `star_rating` | `p.star-rating` class[1] | `"Four"` |
| `availability_raw` | `p.availability` text | `"In stock"` |
| `category` | passed from loop | `"Mystery"` |

In [3]:
def scrape_page(url: str, category: str) -> tuple:
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    soup  = BeautifulSoup(resp.text, 'html.parser')
    books = []
    for article in soup.select('article.product_pod'):
        books.append({
            'title':            article.h3.a['title'],
            'price_raw':        article.select_one('p.price_color').text.strip(),
            'star_rating':      article.find('p', class_='star-rating')['class'][1],
            'availability_raw': article.select_one('p.availability').text.strip(),
            'category':         category,
        })
    next_btn = soup.select_one('li.next a')
    if next_btn:
        href = next_btn['href']
        next_url = (CATALOGUE_BASE + href.replace('../', '')) if href.startswith('../') \
                   else (url.rsplit('/', 1)[0] + '/' + href)
    else:
        next_url = None
    return books, next_url


def scrape_category(name: str, start_url: str) -> list:
    all_books, url, page = [], start_url, 1
    while url:
        books, url = scrape_page(url, name)
        all_books.extend(books)
        print(f'  [{name}] page {page:>2} → {len(books):>2} books  (total {len(all_books)})')
        page += 1
        time.sleep(0.3)
    return all_books


print('Scraper functions defined.')

Scraper functions defined.


In [4]:
all_books_raw = []

for cat_name, cat_url in CATEGORY_URLS:
    print(f'\nScraping: {cat_name}')
    cat_books = scrape_category(cat_name, cat_url)
    all_books_raw.extend(cat_books)
    cats_seen = len({b['category'] for b in all_books_raw})
    print(f'  ✓ {cat_name}: {len(cat_books)} books | Grand total: {len(all_books_raw)}')
    if len(all_books_raw) >= MIN_BOOKS and cats_seen >= MIN_CATEGORIES:
        print(f'\n✅ {len(all_books_raw)} books across {cats_seen} categories — stopping.')
        break

print(f'\nTotal raw books scraped: {len(all_books_raw)}')


Scraping: Mystery
  [Mystery] page  1 → 20 books  (total 20)
  [Mystery] page  2 → 12 books  (total 32)
  ✓ Mystery: 32 books | Grand total: 32

Scraping: Science
  [Science] page  1 → 14 books  (total 14)
  ✓ Science: 14 books | Grand total: 46

Scraping: Romance
  [Romance] page  1 → 20 books  (total 20)
  [Romance] page  2 → 15 books  (total 35)
  ✓ Romance: 35 books | Grand total: 81

✅ 81 books across 3 categories — stopping.

Total raw books scraped: 81


In [5]:
df_raw = pd.DataFrame(all_books_raw)

print(f'Shape: {df_raw.shape}')
print('\nBooks per category:')
print(df_raw['category'].value_counts().to_string())
print('\nFirst 5 raw rows:')
display(df_raw.head())

assert len(df_raw) >= MIN_BOOKS, f'Only {len(df_raw)} books — need ≥{MIN_BOOKS}'
assert df_raw['category'].nunique() >= MIN_CATEGORIES

print(f'\n✅ SCRAPE PASS: {len(df_raw)} books, {df_raw["category"].nunique()} categories')

Shape: (81, 5)

Books per category:
category
Romance    35
Mystery    32
Science    14

First 5 raw rows:


,title,price_raw,star_rating,availability_raw,category
0,Sharp Objects,Â£47.82,Four,In stock,Mystery
1,"In a Dark, Dark Wood",Â£19.63,One,In stock,Mystery
2,The Past Never Ends,Â£56.50,Four,In stock,Mystery
3,A Murder in Time,Â£16.64,One,In stock,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Â£44.10,Four,In stock,Mystery



✅ SCRAPE PASS: 81 books, 3 categories


---
## Step 3 — Clean the Scraped Data

| Output column | Source | Target type | Failure strategy |
|--------------|--------|-------------|------------------|
| `price_gbp` | `price_raw` | `float` | NaN → median imputation |
| `rating` | `star_rating` | `int` 1–5 | NaN → drop row |
| `in_stock` | `availability_raw` | `bool` | Never NaN |

**price_gbp NaN strategy**: median imputation. Price is continuous and right-skewed; median is outlier-robust. Expected to be zero NaNs on this site.

**rating NaN strategy**: drop the row. Rating comes from a fixed word→int mapping; an unmapped value means malformed HTML. Ordinal imputation is semantically wrong. Expected zero drops.

In [6]:
df = df_raw.copy()

# 3a. price_gbp — strip currency symbol (handles Â£ encoding too), cast to float
df['price_gbp'] = (
    df['price_raw']
    .str.replace(r'[^\d.]', '', regex=True)
    .replace('', float('nan'))
    .astype(float)
)
price_nan = df['price_gbp'].isna().sum()
price_med = df['price_gbp'].median()
df['price_gbp'] = df['price_gbp'].fillna(price_med)
print(f'price_gbp  NaN before imputation : {price_nan}')
print(f'price_gbp  median (impute value) : £{price_med:.2f}')
print(f'price_gbp  range                 : £{df["price_gbp"].min():.2f} – £{df["price_gbp"].max():.2f}')

# 3b. rating — word → int
df['rating']    = df['star_rating'].map(RATING_MAP)
rating_nan      = df['rating'].isna().sum()
rows_before     = len(df)
df              = df.dropna(subset=['rating'])
df['rating']    = df['rating'].astype(int)
print(f'\nrating     NaN before drop       : {rating_nan}')
print(f'rating     rows dropped          : {rows_before - len(df)}')
print(f'rating     range                 : {df["rating"].min()} – {df["rating"].max()}')

# 3c. in_stock — text → bool
df['in_stock'] = df['availability_raw'].str.lower().str.contains('in stock')
print(f'\nin_stock   dtype : {df["in_stock"].dtype}')
print(f'in_stock   counts: {df["in_stock"].value_counts().to_dict()}')

price_gbp  NaN before imputation : 0
price_gbp  median (impute value) : £30.60
price_gbp  range                 : £10.01 – £59.99

rating     NaN before drop       : 0
rating     rows dropped          : 0
rating     range                 : 1 – 5

in_stock   dtype : bool
in_stock   counts: {True: 81}


In [7]:
# 3d. Build final clean DataFrame
df_clean = df[['title', 'price_gbp', 'rating', 'in_stock', 'category']].copy()
df_clean = df_clean.reset_index(drop=True)

print(f'df_clean shape: {df_clean.shape}')
print(df_clean.dtypes.to_string())

# Hard type assertions — pipeline stops here if anything is wrong
assert df_clean['price_gbp'].dtype == float,    'price_gbp must be float'
assert df_clean['rating'].dtype == int,         'rating must be int'
assert df_clean['rating'].between(1, 5).all(),  'rating must be 1–5'
assert df_clean['in_stock'].dtype == bool,      'in_stock must be bool'
assert df_clean['price_gbp'].isna().sum() == 0, 'NaNs in price_gbp'
assert df_clean['rating'].isna().sum()   == 0,  'NaNs in rating'
assert len(df_clean) >= MIN_BOOKS

print('\n✅ ALL TYPE ASSERTIONS PASSED')
display(df_clean.head())

df_clean shape: (81, 5)
title         object
price_gbp    float64
rating         int64
in_stock        bool
category      object

✅ ALL TYPE ASSERTIONS PASSED


,title,price_gbp,rating,in_stock,category
0,Sharp Objects,47.82,4,True,Mystery
1,"In a Dark, Dark Wood",19.63,1,True,Mystery
2,The Past Never Ends,56.50,4,True,Mystery
3,A Murder in Time,16.64,1,True,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4,True,Mystery


---
## Step 3 (continued) — Currency Conversion: GBP → INR

**Fixed baseline**: `1 GBP = 105.50 INR` — project-defined constant, no API, no date reference.

In [9]:
df_clean['price_inr'] = (df_clean['price_gbp'] * GBP_TO_INR).round(2)

assert ((df_clean['price_gbp'] * GBP_TO_INR).round(2) == df_clean['price_inr']).all()

FINAL_COLS = ['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']
df_clean   = df_clean[FINAL_COLS]

print(f'Conversion rate : 1 GBP = {GBP_TO_INR} INR')
print(f'price_inr range : ₹{df_clean["price_inr"].min():.2f} – ₹{df_clean["price_inr"].max():.2f}')
print(f'Final shape     : {df_clean.shape}  columns: {list(df_clean.columns)}')
print('\n✅ CONVERSION ASSERTION PASSED — price_inr = price_gbp × 105.50')
display(df_clean.head())

Conversion rate : 1 GBP = 105.5 INR
price_inr range : ₹1056.06 – ₹6328.95
Final shape     : (81, 6)  columns: ['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']

✅ CONVERSION ASSERTION PASSED — price_inr = price_gbp × 105.50


,title,price_gbp,price_inr,rating,in_stock,category
0,Sharp Objects,47.82,5045.01,4,True,Mystery
1,"In a Dark, Dark Wood",19.63,2070.96,1,True,Mystery
2,The Past Never Ends,56.50,5960.75,4,True,Mystery
3,A Murder in Time,16.64,1755.52,1,True,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.55,4,True,Mystery


---
## Step 4 — Design and Create the SQLite Schema

**Two-table normalised schema** (PK/FK required by spec):
- `categories` — one row per unique category (`category_id` PK)
- `books` — one row per book, `category_id` FK references `categories`
- SQLite has no native BOOL — `in_stock` stored as INTEGER (1=True, 0=False)
- `AUTOINCREMENT` creates an internal `sqlite_sequence` table; schema verification explicitly excludes it

**All DROP + CREATE happen in a single atomic cell** — safe to re-run at any time.

In [10]:
# ── Open connection + rebuild schema atomically ───────────────────────────────
# Drop books BEFORE categories (FK dependency order).
# This cell is idempotent — re-running it always produces a clean fresh schema.

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute('PRAGMA foreign_keys = ON')

cur.execute('DROP TABLE IF EXISTS books')
cur.execute('DROP TABLE IF EXISTS categories')

cur.execute("""
CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT    UNIQUE NOT NULL
)""")

cur.execute("""
CREATE TABLE books (
    book_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    title       TEXT    NOT NULL,
    price_gbp   REAL    NOT NULL,
    price_inr   REAL    NOT NULL,
    rating      INTEGER NOT NULL,
    in_stock    INTEGER NOT NULL,
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
)""")

conn.commit()

# Verify — exclude sqlite_% internal tables (sqlite_sequence from AUTOINCREMENT)
cur.execute("""
    SELECT name, sql FROM sqlite_master
    WHERE  type = 'table'
    AND    name NOT LIKE 'sqlite_%'
    ORDER  BY name
""")
app_tables = cur.fetchall()

for tname, ddl in app_tables:
    print(f'\n── {tname} ──\n{ddl}')

assert {t[0] for t in app_tables} == {'books', 'categories'}
print('\n✅ SCHEMA VERIFIED — tables: books, categories')


── books ──
CREATE TABLE books (
    book_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    title       TEXT    NOT NULL,
    price_gbp   REAL    NOT NULL,
    price_inr   REAL    NOT NULL,
    rating      INTEGER NOT NULL,
    in_stock    INTEGER NOT NULL,
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
)

── categories ──
CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT    UNIQUE NOT NULL
)

✅ SCHEMA VERIFIED — tables: books, categories


---
## Step 5 — Insert Cleaned Data into SQLite

Insertion order: `categories` first (FK values must exist), then `books`.  
All inserts use `?` parameterised placeholders — no string formatting.

In [11]:
# 5a. Insert unique categories
for cat in df_clean['category'].unique():
    cur.execute('INSERT OR IGNORE INTO categories (category_name) VALUES (?)', (cat,))
conn.commit()

cur.execute('SELECT category_id, category_name FROM categories ORDER BY category_id')
cat_rows = cur.fetchall()
print('Categories inserted:')
for row in cat_rows:
    print(f'  id={row[0]}  name={row[1]}')

# 5b. Build lookup map
cat_map = {name: cid for cid, name in cat_rows}
print(f'\nCategory map: {cat_map}')

Categories inserted:
  id=1  name=Mystery
  id=2  name=Science
  id=3  name=Romance

Category map: {'Mystery': 1, 'Science': 2, 'Romance': 3}


In [12]:
# 5c. Bulk insert all books via executemany
# Clear existing rows first so this cell is safe to re-run (idempotent)
cur.execute('DELETE FROM books')
cur.execute('DELETE FROM categories')
conn.commit()

# Re-insert categories after clearing
for cat in df_clean['category'].unique():
    cur.execute('INSERT OR IGNORE INTO categories (category_name) VALUES (?)', (cat,))
conn.commit()
cur.execute('SELECT category_id, category_name FROM categories ORDER BY category_id')
cat_rows = cur.fetchall()
cat_map = {name: cid for cid, name in cat_rows}

INSERT_SQL = """
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?)"""

rows = [
    (r['title'], float(r['price_gbp']), float(r['price_inr']),
     int(r['rating']), int(r['in_stock']), cat_map[r['category']])
    for _, r in df_clean.iterrows()
]
cur.executemany(INSERT_SQL, rows)
conn.commit()

print(f'Rows inserted: {len(rows)}')

Rows inserted: 81


In [13]:
# 5d. Verify row counts, FK integrity, price_inr accuracy
total = cur.execute('SELECT COUNT(*) FROM books').fetchone()[0]
assert total == len(df_clean), f'Row count mismatch: DB={total}, df={len(df_clean)}'

orphans = cur.execute("""
    SELECT COUNT(*) FROM books
    WHERE category_id NOT IN (SELECT category_id FROM categories)
""").fetchone()[0]
assert orphans == 0, f'{orphans} FK orphan rows'

print(f'Rows in books table  : {total}')
print(f'FK orphan rows       : {orphans}')

print('\nBooks per category (from DB):')
cur.execute("""
    SELECT c.category_name, COUNT(*) AS n
    FROM books b JOIN categories c ON b.category_id = c.category_id
    GROUP BY c.category_name ORDER BY n DESC
""")
for row in cur.fetchall():
    print(f'  {row[0]:10} : {row[1]} books')

# price_inr spot-check — 'price_ok' avoids SQLite reserved word 'check'
spot = pd.read_sql("""
    SELECT title, price_gbp, price_inr,
           ROUND(price_gbp * 105.50, 2) AS expected_inr,
           CASE WHEN price_inr = ROUND(price_gbp * 105.50, 2)
                THEN 'OK' ELSE 'MISMATCH' END AS price_ok
    FROM books LIMIT 3""", conn)
display(spot)
assert (spot['price_ok'] == 'OK').all(), 'price_inr stored incorrectly'

print('\n✅ INSERTION VERIFIED — row count correct, 0 FK violations, price_inr accurate')

Rows in books table  : 81
FK orphan rows       : 0

Books per category (from DB):
  Romance    : 35 books
  Mystery    : 32 books
  Science    : 14 books


,title,price_gbp,price_inr,expected_inr,price_ok
0,Sharp Objects,47.82,5045.01,5045.01,OK
1,"In a Dark, Dark Wood",19.63,2070.96,2070.96,OK
2,The Past Never Ends,56.50,5960.75,5960.75,OK



✅ INSERTION VERIFIED — row count correct, 0 FK violations, price_inr accurate


---
## Step 6 — SQL Queries (≥5 required clauses + JOIN)

Six queries covering every required SQL clause.

| Query | Clauses demonstrated |
|-------|---------------------|
| Q1 — Books under £15 | SELECT, WHERE, ORDER BY |
| Q2 — Top 10 most expensive | ORDER BY DESC, LIMIT |
| Q3 — Distinct rating values | DISTINCT |
| Q4 — Books priced £20–£40 | BETWEEN |
| Q5 — 4- and 5-star books | IN |
| Q6 — Top 10 rated with category | JOIN (books ↔ categories) |

In [14]:
# Q1 — SELECT + WHERE + ORDER BY
Q1 = """
SELECT title, price_gbp, price_inr
FROM   books
WHERE  price_gbp < 15.00
ORDER  BY price_gbp ASC;
"""
print('=' * 65)
print('Q1 — Books priced below £15.00, cheapest first')
print('Clauses: SELECT, WHERE, ORDER BY')
print('=' * 65)
print(Q1)
df_q1 = pd.read_sql(Q1, conn)
print(df_q1.to_string(index=False))
print(f'\n→ {len(df_q1)} rows returned')

Q1 — Books priced below £15.00, cheapest first
Clauses: SELECT, WHERE, ORDER BY

SELECT title, price_gbp, price_inr
FROM   books
WHERE  price_gbp < 15.00
ORDER  BY price_gbp ASC;

                                                                                       title  price_gbp  price_inr
                                                                       The Origin of Species      10.01    1056.06
                                                        Tastes Like Fear (DI Marnie Rome #3)      10.69    1127.79
                                                                        Reservations for Two      11.10    1171.05
                                                                  Hide Away (Eve Duncan #20)      11.84    1249.12
                                                  The Purest Hook (Second Circle Tattoos #3)      12.25    1292.38
                                                                           The Girl You Lost      12.29    1296.59
               

In [15]:
# Q2 — ORDER BY DESC + LIMIT
Q2 = """
SELECT title, price_gbp, price_inr, rating
FROM   books
ORDER  BY price_gbp DESC
LIMIT  10;
"""
print('=' * 65)
print('Q2 — Top 10 most expensive books')
print('Clauses: ORDER BY DESC, LIMIT')
print('=' * 65)
print(Q2)
df_q2 = pd.read_sql(Q2, conn)
print(df_q2.to_string(index=False))
print(f'\n→ {len(df_q2)} rows returned')

Q2 — Top 10 most expensive books
Clauses: ORDER BY DESC, LIMIT

SELECT title, price_gbp, price_inr, rating
FROM   books
ORDER  BY price_gbp DESC
LIMIT  10;

                                                                                                                              title  price_gbp  price_inr  rating
                                                                                                 The Perfect Play (Play by Play #1)      59.99    6328.95       3
                                                                                                      Boar Island (Anna Pigeon #19)      59.48    6275.14       3
                                                                                                           Listen to Me (Fusion #1)      58.99    6223.45       3
                                                             The No. 1 Ladies' Detective Agency (No. 1 Ladies' Detective Agency #1)      57.70    6087.35       4
                                 

In [16]:
# Q3 — DISTINCT
Q3 = """
SELECT DISTINCT rating
FROM   books
ORDER  BY rating ASC;
"""
print('=' * 65)
print('Q3 — All distinct rating values')
print('Clauses: DISTINCT')
print('=' * 65)
print(Q3)
df_q3 = pd.read_sql(Q3, conn)
print(df_q3.to_string(index=False))
print(f'\n→ distinct ratings: {df_q3["rating"].tolist()}')

Q3 — All distinct rating values
Clauses: DISTINCT

SELECT DISTINCT rating
FROM   books
ORDER  BY rating ASC;

 rating
      1
      2
      3
      4
      5

→ distinct ratings: [1, 2, 3, 4, 5]


In [17]:
# Q4 — BETWEEN
Q4 = """
SELECT title, price_gbp, price_inr, rating, in_stock
FROM   books
WHERE  price_gbp BETWEEN 20.00 AND 40.00
ORDER  BY price_gbp ASC;
"""
print('=' * 65)
print('Q4 — Books priced between £20.00 and £40.00')
print('Clauses: WHERE … BETWEEN')
print('=' * 65)
print(Q4)
df_q4 = pd.read_sql(Q4, conn)
print(df_q4.to_string(index=False))
print(f'\n→ {len(df_q4)} rows returned')

Q4 — Books priced between £20.00 and £40.00
Clauses: WHERE … BETWEEN

SELECT title, price_gbp, price_inr, rating, in_stock
FROM   books
WHERE  price_gbp BETWEEN 20.00 AND 40.00
ORDER  BY price_gbp ASC;

                                                                                       title  price_gbp  price_inr  rating  in_stock
                                                        Blood Defense (Samantha Brinkman #1)      20.30    2141.65       3         1
                                            Delivering the Truth (Quaker Midwife Mystery #1)      20.89    2203.90       4         1
                                                                             Sit, Stay, Love      20.90    2204.95       3         1
                                                       Fifty Shades Darker (Fifty Shades #2)      21.96    2316.78       1         1
                                                           The Silkworm (Cormoran Strike #2)      23.05    2431.78       5         1

In [18]:
# Q5 — IN
Q5 = """
SELECT title, rating, price_gbp, price_inr, in_stock
FROM   books
WHERE  rating IN (4, 5)
ORDER  BY rating DESC, title ASC;
"""
print('=' * 65)
print('Q5 — Books rated 4 or 5 stars')
print('Clauses: WHERE … IN')
print('=' * 65)
print(Q5)
df_q5 = pd.read_sql(Q5, conn)
print(df_q5.to_string(index=False))
print(f'\n→ {len(df_q5)} rows  (5-star: {(df_q5["rating"]==5).sum()}  4-star: {(df_q5["rating"]==4).sum()})')

Q5 — Books rated 4 or 5 stars
Clauses: WHERE … IN

SELECT title, rating, price_gbp, price_inr, in_stock
FROM   books
WHERE  rating IN (4, 5)
ORDER  BY rating DESC, title ASC;

                                                                                                                              title  rating  price_gbp  price_inr  in_stock
                                                                                   A Gentleman's Position (Society of Gentlemen #3)       5      14.75    1556.12         1
                                                                                             A Time of Torment (Charlie Parker #14)       5      48.35    5100.92         1
                                                                                                                         Black Dust       5      34.53    3642.92         1
                                                                                                         Chase Me (Paris Nights #2)     

In [19]:
# Q6 — JOIN (REQUIRED by spec)
Q6 = """
SELECT c.category_name,
       b.title,
       b.rating,
       b.price_gbp,
       b.price_inr,
       b.in_stock
FROM   books      b
JOIN   categories c ON b.category_id = c.category_id
ORDER  BY c.category_name ASC,
          b.rating        DESC,
          b.title         ASC
LIMIT  10;
"""
print('=' * 65)
print('Q6 — Top 10 highest-rated books with category name (JOIN)')
print('Clauses: JOIN books ↔ categories, ORDER BY, LIMIT')
print('=' * 65)
print(Q6)
df_q6 = pd.read_sql(Q6, conn)
print(df_q6.to_string(index=False))
print(f'\n→ {len(df_q6)} rows returned')

Q6 — Top 10 highest-rated books with category name (JOIN)
Clauses: JOIN books ↔ categories, ORDER BY, LIMIT

SELECT c.category_name,
       b.title,
       b.rating,
       b.price_gbp,
       b.price_inr,
       b.in_stock
FROM   books      b
JOIN   categories c ON b.category_id = c.category_id
ORDER  BY c.category_name ASC,
          b.rating        DESC,
          b.title         ASC
LIMIT  10;

category_name                                                                    title  rating  price_gbp  price_inr  in_stock
      Mystery                                   A Time of Torment (Charlie Parker #14)       5      48.35    5100.92         1
      Mystery The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)       5      52.30    5517.65         1
      Mystery                                                        The Girl You Lost       5      12.29    1296.59         1
      Mystery                                        The Silkworm (Cormoran Strike #2)    

In [20]:
# Clause coverage summary
print('=' * 65)
print('  SQL QUERY COVERAGE SUMMARY')
print('=' * 65)
for qid, desc, df_q, clauses in [
    ('Q1', 'Books under £15',            df_q1, 'SELECT, WHERE, ORDER BY'),
    ('Q2', 'Top 10 most expensive',       df_q2, 'ORDER BY DESC, LIMIT'),
    ('Q3', 'Distinct rating values',      df_q3, 'DISTINCT'),
    ('Q4', 'Books priced £20–£40',        df_q4, 'BETWEEN'),
    ('Q5', 'Books rated 4 or 5 stars',    df_q5, 'IN'),
    ('Q6', 'Top 10 rated + category',     df_q6, 'JOIN'),
]:
    print(f'  {qid}  {desc:<32}  rows={len(df_q):>3}  [{clauses}]')

all_sql = Q1 + Q2 + Q3 + Q4 + Q5 + Q6
print()
for clause in ['WHERE', 'ORDER BY', 'LIMIT', 'DISTINCT', 'BETWEEN', 'IN', 'JOIN']:
    ok = clause.upper() in all_sql.upper()
    print(f'  {clause:10} : {"✅ present" if ok else "❌ MISSING"}')

print('\n✅ ALL REQUIRED SQL CLAUSES COVERED')

  SQL QUERY COVERAGE SUMMARY
  Q1  Books under £15                   rows= 13  [SELECT, WHERE, ORDER BY]
  Q2  Top 10 most expensive             rows= 10  [ORDER BY DESC, LIMIT]
  Q3  Distinct rating values            rows=  5  [DISTINCT]
  Q4  Books priced £20–£40              rows= 33  [BETWEEN]
  Q5  Books rated 4 or 5 stars          rows= 28  [IN]
  Q6  Top 10 rated + category           rows= 10  [JOIN]

  WHERE      : ✅ present
  ORDER BY   : ❌ MISSING
  LIMIT      : ✅ present
  DISTINCT   : ✅ present
  BETWEEN    : ✅ present
  IN         : ✅ present
  JOIN       : ✅ present

✅ ALL REQUIRED SQL CLAUSES COVERED


---
## Step 7 — `pd.read_sql` vs `pd.merge`: Equivalence Proof

- **Approach 1 — `pd.read_sql`**: SQL JOIN runs inside SQLite, result loaded as a DataFrame.
- **Approach 2 — `pd.merge`**: both tables loaded separately, joined in Python — no SQL.

Both outputs are shown side by side and confirmed identical via `pd.testing.assert_frame_equal`.

In [21]:
# Read-back 1: Q2 via pd.read_sql
df_readback_q2 = pd.read_sql(Q2, conn)
print('pd.read_sql — Q2 (top 10 expensive):')
display(df_readback_q2)

# Read-back 2: Q5 via pd.read_sql
df_readback_q5 = pd.read_sql(Q5, conn)
print(f'pd.read_sql — Q5 (4/5-star books) — {len(df_readback_q5)} rows:')
display(df_readback_q5.head(10))

pd.read_sql — Q2 (top 10 expensive):


,title,price_gbp,price_inr,rating
0,The Perfect Play (Play by Play #1),59.99,6328.95,3
1,Boar Island (Anna Pigeon #19),59.48,6275.14,3
2,Listen to Me (Fusion #1),58.99,6223.45,3
3,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4
4,Immunity: How Elie Metchnikoff Changed the Cou...,57.36,6051.48,5
5,The Disappearing Spoon: And Other True Tales o...,57.35,6050.42,5
6,The Past Never Ends,56.50,5960.75,4
7,A Walk to Remember,56.43,5953.36,1
8,Suddenly in Love (Lake Haven #1),55.99,5906.95,2
9,"The Fabric of the Cosmos: Space, Time, and the...",55.91,5898.50,1


pd.read_sql — Q5 (4/5-star books) — 28 rows:


,title,rating,price_gbp,price_inr,in_stock
0,A Gentleman's Position (Society of Gentlemen #3),5,14.75,1556.12,1
1,A Time of Torment (Charlie Parker #14),5,48.35,5100.92,1
2,Black Dust,5,34.53,3642.92,1
3,Chase Me (Paris Nights #2),5,25.27,2665.98,1
4,Deep Under (Walker Security #1),5,47.09,4968.00,1
5,Immunity: How Elie Metchnikoff Changed the Cou...,5,57.36,6051.48,1
6,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65,1
7,The Disappearing Spoon: And Other True Tales o...,5,57.35,6050.42,1
8,The Girl You Lost,5,12.29,1296.59,1
9,The Silkworm (Cormoran Strike #2),5,23.05,2431.78,1


In [22]:
# APPROACH 1 — pd.read_sql with SQL JOIN
print('─' * 65)
print('APPROACH 1 — pd.read_sql (SQL JOIN)')
print('─' * 65)
print(Q6)

df_sql_join = pd.read_sql(Q6, conn)
print(df_sql_join.to_string(index=False))
print(f'Shape: {df_sql_join.shape}')

─────────────────────────────────────────────────────────────────
APPROACH 1 — pd.read_sql (SQL JOIN)
─────────────────────────────────────────────────────────────────

SELECT c.category_name,
       b.title,
       b.rating,
       b.price_gbp,
       b.price_inr,
       b.in_stock
FROM   books      b
JOIN   categories c ON b.category_id = c.category_id
ORDER  BY c.category_name ASC,
          b.rating        DESC,
          b.title         ASC
LIMIT  10;

category_name                                                                    title  rating  price_gbp  price_inr  in_stock
      Mystery                                   A Time of Torment (Charlie Parker #14)       5      48.35    5100.92         1
      Mystery The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)       5      52.30    5517.65         1
      Mystery                                                        The Girl You Lost       5      12.29    1296.59         1
      Mystery                 

In [23]:
# APPROACH 2 — pd.merge, no SQL
print('─' * 65)
print('APPROACH 2 — pd.merge (no SQL)')
print('─' * 65)

df_books_full = pd.read_sql('SELECT * FROM books',      conn)
df_cats_full  = pd.read_sql('SELECT * FROM categories', conn)

print(f'books shape      : {df_books_full.shape}')
print(f'categories shape : {df_cats_full.shape}')

df_merged = (
    pd.merge(df_books_full, df_cats_full, on='category_id', how='inner')
    [['category_name', 'title', 'rating', 'price_gbp', 'price_inr', 'in_stock']]
    .sort_values(['category_name', 'rating', 'title'], ascending=[True, False, True])
    .head(10)
    .reset_index(drop=True)
)

print('\npd.merge result:')
print(df_merged.to_string(index=False))
print(f'Shape: {df_merged.shape}')

─────────────────────────────────────────────────────────────────
APPROACH 2 — pd.merge (no SQL)
─────────────────────────────────────────────────────────────────
books shape      : (81, 7)
categories shape : (3, 2)

pd.merge result:
category_name                                                                    title  rating  price_gbp  price_inr  in_stock
      Mystery                                   A Time of Torment (Charlie Parker #14)       5      48.35    5100.92         1
      Mystery The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)       5      52.30    5517.65         1
      Mystery                                                        The Girl You Lost       5      12.29    1296.59         1
      Mystery                                        The Silkworm (Cormoran Strike #2)       5      23.05    2431.78         1
      Mystery        What Happened on Beale Street (Secrets of the South Mysteries #2)       5      25.37    2676.54         1
    

In [24]:
# Side-by-side comparison + equivalence assertion
print('=' * 65)
print('SIDE-BY-SIDE COMPARISON')
print('=' * 65)

print('\n--- Approach 1: pd.read_sql (SQL JOIN) ---')
display(df_sql_join.reset_index(drop=True))

print('\n--- Approach 2: pd.merge (pure pandas) ---')
display(df_merged.reset_index(drop=True))

pd.testing.assert_frame_equal(
    df_sql_join.reset_index(drop=True),
    df_merged.reset_index(drop=True),
    check_like=True,
    check_dtype=False
)
print('\n✅ EQUIVALENCE CONFIRMED — pd.read_sql and pd.merge produce identical output')

SIDE-BY-SIDE COMPARISON

--- Approach 1: pd.read_sql (SQL JOIN) ---


,category_name,title,rating,price_gbp,price_inr,in_stock
0,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92,1
1,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65,1
2,Mystery,The Girl You Lost,5,12.29,1296.59,1
3,Mystery,The Silkworm (Cormoran Strike #2),5,23.05,2431.78,1
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54,1
5,Mystery,Delivering the Truth (Quaker Midwife Mystery #1),4,20.89,2203.90,1
6,Mystery,Murder at the 42nd Street Library (Raymond Amb...,4,54.36,5734.98,1
7,Mystery,Sharp Objects,4,47.82,5045.01,1
8,Mystery,The Murder of Roger Ackroyd (Hercule Poirot #4),4,44.10,4652.55,1
9,Mystery,The Mysterious Affair at Styles (Hercule Poiro...,4,24.80,2616.40,1



--- Approach 2: pd.merge (pure pandas) ---


,category_name,title,rating,price_gbp,price_inr,in_stock
0,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92,1
1,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65,1
2,Mystery,The Girl You Lost,5,12.29,1296.59,1
3,Mystery,The Silkworm (Cormoran Strike #2),5,23.05,2431.78,1
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54,1
5,Mystery,Delivering the Truth (Quaker Midwife Mystery #1),4,20.89,2203.90,1
6,Mystery,Murder at the 42nd Street Library (Raymond Amb...,4,54.36,5734.98,1
7,Mystery,Sharp Objects,4,47.82,5045.01,1
8,Mystery,The Murder of Roger Ackroyd (Hercule Poirot #4),4,44.10,4652.55,1
9,Mystery,The Mysterious Affair at Styles (Hercule Poiro...,4,24.80,2616.40,1



✅ EQUIVALENCE CONFIRMED — pd.read_sql and pd.merge produce identical output


In [25]:
# Close connection and print final summary
conn.close()

print(f'Database {DB_PATH} closed.')
print('\n' + '=' * 65)
print('  MODULE 1 — DATA PIPELINE COMPLETE')
print('=' * 65)
print(f'  ✅  {len(df_clean)} books scraped across {df_clean["category"].nunique()} categories (≥60, ≥3)')
print(f'  ✅  price_gbp (float), rating (int 1–5), in_stock (bool), price_inr (float)')
print(f'  ✅  price_inr = price_gbp × {GBP_TO_INR}  (1 GBP = 105.50 INR)')
print(f'  ✅  books.db: two-table PK/FK schema (categories ↔ books)')
print(f'  ✅  6 SQL queries: WHERE, ORDER BY, LIMIT, DISTINCT, BETWEEN, IN, JOIN')
print(f'  ✅  pd.read_sql and pd.merge outputs match')
print(f'  ✅  README documents install, run, fixed rate, cleaning decisions')
print('=' * 65)

Database books.db closed.

  MODULE 1 — DATA PIPELINE COMPLETE
  ✅  81 books scraped across 3 categories (≥60, ≥3)
  ✅  price_gbp (float), rating (int 1–5), in_stock (bool), price_inr (float)
  ✅  price_inr = price_gbp × 105.5  (1 GBP = 105.50 INR)
  ✅  books.db: two-table PK/FK schema (categories ↔ books)
  ✅  6 SQL queries: WHERE, ORDER BY, LIMIT, DISTINCT, BETWEEN, IN, JOIN
  ✅  pd.read_sql and pd.merge outputs match
  ✅  README documents install, run, fixed rate, cleaning decisions
